## Selected Tokens

In [1]:
from datasets import load_dataset, Dataset, load_from_disk
from transformers import AutoTokenizer, GPT2Config, T5Config
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer, GPT2LMHeadModel, AutoModel
from transformers import Pipeline, TextClassificationPipeline
from transformers import DataCollatorForLanguageModeling, TrainerCallback
import math
import argparse
from tqdm import tqdm
import torch
import torch.nn as nn
import random
import os
import pickle
import re
from matplotlib import pyplot as plt
from torch.utils.data import DataLoader, SequentialSampler

from newModelsT5Simplified import T5ModelDecoderCacheSelector
from utilsFP import FixPredictionPipelineGenerator, BiLSTMRegression
from utilsLM import customTrainer, customCollator, process_wiki
import logging

dataset = 'wiki'

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
tokenizer_name = 't5-small'

xl_cache_size = 128
fix_cache_size = 384

if dataset == 'wiki':
    model_name = "..."
    data_path = f'./LMdatasets/processed-wiki2-{tokenizer_name}-32-512-fix12'
    valid_batch_size = 8

elif dataset == 'dog':
    model_name = "..."
    data_path = f'./LMdatasets/processed-dog3-{tokenizer_name}-8-512-fix12'
    valid_batch_size = 4

elif dataset == 'pg19':
    model_name = "..."
    data_path = f'./LMdatasets/processed-{tokenizer_name}-32-512-fix12'
    valid_batch_size = 8

tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, cache_dir='./cache/models', model_max_length=2048)

model = T5ModelDecoderCacheSelector.from_pretrained(model_name, block_size=512, xl_cache_size=xl_cache_size, fix_cache_size=fix_cache_size, \
                                    threshold=9, new_fix_cache_per_step=0, selector_cache_size=128, \
                                    selector_cache_num=3, rmD=True, L0_lambda=0.01, tokenizer=tokenizer)


block_size = 512

tokenizer.pad_token = tokenizer.eos_token


print('loading dataset...')
tokenized_dataset = load_from_disk(data_path)


data_collator = customCollator(tokenizer=tokenizer)

valid_dataset = tokenized_dataset["validation"]
valid_loader = DataLoader(
            valid_dataset,
            sampler=SequentialSampler(valid_dataset),
            batch_size=valid_batch_size,
            collate_fn=data_collator
        )



Discovered apex.normalization.FusedRMSNorm - will use it instead of T5LayerNorm
Discovered apex.normalization.FusedRMSNorm - will use it instead of T5LayerNorm
params:	 xl_cache 128 fix_cache 384 0 selector 128 3 0.01 True
loading dataset...


In [3]:
with torch.no_grad():
    model.to(device)
    model.reset_cache()
    model.eval()
    count = 0
    for inputs in tqdm(valid_loader):
        inputs = {k: v.to(device) for k, v in inputs.items()}
        outputs = model(**inputs)
        if count > 10:
            break
        count += 1



empty cache...


 15%|█▌        | 11/72 [00:07<00:41,  1.48it/s]


In [4]:
from IPython.core.display import display, HTML
def highlighter(word, value):
    return '<span style="background-color:rgba(40,156,206,%.2f)">'%(value) +word+ '</span>' 

for i in range(model.config.num_layers):
    layer_idx = i
    batch_idx = 2
    info = model.decoder.block[layer_idx].layer[0].SelfAttention.pass_info

    ids = info[f'{batch_idx}-ids'].tolist()

    mask1 = info[f'{batch_idx}-mask1']
    mask2 = mask1.clone()
    mask2[mask1] = info[f'{batch_idx}-mask2']
    total_mask = (mask1.float() + mask2.float())/2

    html = '<span style="background-color:rgba(246,106,100, 1.0)">' +f'Layer {layer_idx}: '+ '</span>' + ''.join([highlighter(token.replace('▁', ' '), p) for p, token in zip(total_mask.tolist(), tokenizer.convert_ids_to_tokens(ids))])

    display(HTML(html))

/tmp/ipykernel_1838866/2150380150.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


### show eye fixation

In [32]:
from IPython.core.display import display, HTML
def highlighter(word, value):
    return '<span style="background-color:rgba(40,156,206,%.2f)">'%(value) +word+ '</span>' 


count = 0
for inputs in tqdm(valid_loader):
    if count > 10:
        break
    count += 1


batch_idx = 3
ids = inputs['input_ids'][batch_idx].tolist()
total_mask = (inputs['fix_duration'][batch_idx] >= 9).float().tolist()

html = ''.join([highlighter(token.replace('▁', ' '), p) for p, token in zip(total_mask, tokenizer.convert_ids_to_tokens(ids))])

display(HTML(html))

/tmp/ipykernel_958453/2238839422.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML
 15%|█▌        | 11/72 [00:00<00:00, 76.51it/s]


## Loss Difference with respect to puncs

In [1]:
from datasets import load_dataset, Dataset, load_from_disk
from transformers import AutoTokenizer, GPT2Config, T5Config
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer, GPT2LMHeadModel, AutoModel
from transformers import Pipeline, TextClassificationPipeline
from transformers import DataCollatorForLanguageModeling, TrainerCallback
import math
import argparse
from tqdm import tqdm
import torch
import torch.nn as nn
import random
import os
import pickle
import re
from matplotlib import pyplot as plt
from torch.utils.data import DataLoader, SequentialSampler

from newModelsT5Simplified import T5ModelDecoderCacheSelector
from newModelsT5Simplifiedtemp import T5ModelDecoderCacheSelectortemp
from utilsLM import customTrainer, customCollator, process_wiki
import logging

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
tokenizer_name = 't5-small'

dataset = 'pg19'
xl_cache_size = 128
fix_cache_size = 384

if dataset == 'dog':
    model_name = "..."
    data_path = f'./LMdatasets/processed-dog3-{tokenizer_name}-8-512-fix12'
    valid_batch_size = 4
elif dataset == 'pg19':
    model_name = "..."
    data_path = f'./LMdatasets/processed-{tokenizer_name}-32-512-fix12'
    valid_batch_size = 8

tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, cache_dir='./cache/models', model_max_length=2048)


print('loading dataset...')
tokenized_dataset = load_from_disk(data_path)

# config = T5Config.from_pretrained(model_name)
model = T5ModelDecoderCacheSelector.from_pretrained(model_name, block_size=512, xl_cache_size=xl_cache_size, fix_cache_size=fix_cache_size, \
                                    threshold=9, new_fix_cache_per_step=0, selector_cache_size=128, \
                                    selector_cache_num=3, rmD=True, L0_lambda=0.01, tokenizer=tokenizer)

# calculate frequency
# token_freq = torch.zeros(model.config.vocab_size)
# token_idx = torch.arange(model.config.vocab_size).long()
# for i in torch.randperm(len(tokenized_dataset['train'])).tolist(): #[:1000]:
#     for idx in tokenized_dataset['train'][i]['input_ids']:
#         token_freq[idx] += 1

# max_freq = token_freq.max()
# freq_mask = token_idx[token_freq < (max_freq * 0.001)]
# print(max_freq)

# m_vocab = []
# for k, v in tokenizer.get_vocab().items():
#     # if re.search(r'[a-zA-Z\d]', k) is None:
#     #     masked_vocab.append(v)
#     if re.search(r'[A-Z]', k):
#         m_vocab.append(v)
#     # if k=='.' or k==',':
#     #     masked_vocab.append(v)
# masked_vocab = torch.cat([torch.tensor(m_vocab), freq_mask])

masked_vocab = torch.tensor([5,6])

model_masked = T5ModelDecoderCacheSelectortemp.from_pretrained(model_name, block_size=512, xl_cache_size=xl_cache_size, fix_cache_size=fix_cache_size, \
                                    threshold=9, new_fix_cache_per_step=0, selector_cache_size=128, \
                                    selector_cache_num=3, rmD=True, L0_lambda=0.01, tokenizer=tokenizer, masked_vocab=masked_vocab)

block_size = 512

tokenizer.pad_token = tokenizer.eos_token




data_collator = customCollator(tokenizer=tokenizer)

valid_dataset = tokenized_dataset["validation"]
valid_loader = DataLoader(
            valid_dataset,
            sampler=SequentialSampler(valid_dataset),
            batch_size=valid_batch_size,
            collate_fn=data_collator
        )



Discovered apex.normalization.FusedRMSNorm - will use it instead of T5LayerNorm
Discovered apex.normalization.FusedRMSNorm - will use it instead of T5LayerNorm
loading dataset...
params:	 xl_cache 128 fix_cache 384 0 selector 128 3 0.01 True
params:	 xl_cache 128 fix_cache 384 0 selector 128 3 0.01 True


In [10]:
with torch.no_grad():
    step_num = 5
    model.to(device)
    model.reset_cache()
    model.eval()
    
    count = 0
    for inputs in tqdm(valid_loader):
        inputs = {k: v.to(device) for k, v in inputs.items()}
        outputs = model(**inputs)
        if count > step_num:
            break
        count += 1

    model_masked.to(device)
    model_masked.reset_cache()
    model_masked.eval()

    count = 0
    for inputs in tqdm(valid_loader):
        inputs = {k: v.to(device) for k, v in inputs.items()}
        outputs_masked = model_masked(**inputs)
        if count > step_num:
            break
        count += 1
    
    labels = inputs['labels'][..., 1:].contiguous()
    loss = outputs['loss'].view(labels.size())
    loss_masked = outputs_masked['loss'].view(labels.size())
    loss_diff = loss_masked - loss
    labels[labels==-100] = tokenizer.pad_token_id

empty cache...


  4%|▍         | 6/148 [00:00<00:23,  6.04it/s]


empty cache...


  4%|▍         | 6/148 [00:01<00:27,  5.25it/s]


In [11]:
from IPython.core.display import display, HTML
def highlighter(word, value):
    if value > 0:
        return '<span style="background-color:rgba(255,156,0,%.2f)">'%(value) +word+ '</span>' 
    else:
        return '<span style="background-color:rgba(40,206,150,%.2f)">'%(-value) +word+ '</span>' 


batch_idx = 2
ids = labels[batch_idx].tolist()
# print(loss_diff[batch_idx])
scores = (loss_diff[batch_idx])#.clamp(min=-1.0, max=1.0)

html = ''.join([highlighter(token.replace('▁', ' '), s) for s, token in zip(scores.tolist(), tokenizer.convert_ids_to_tokens(ids))])

display(HTML(html))

/tmp/ipykernel_1852749/802735530.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [2]:
with torch.no_grad():

    model.to(device)
    model.reset_cache()
    model.eval()

    model_masked.to(device)
    model_masked.reset_cache()
    model_masked.eval()

    puncs_diff = 0
    puncs_diff_abs = 0
    puncs_num = 0
    after_puncs_diff = 0
    after_puncs_diff_abs = 0
    after_puncs_num = 0

    total_diff = 0
    total_diff_abs = 0
    total_num = 0

    MASKED_VOCAB = torch.tensor([5,6]).to(device)
    for inputs in tqdm(valid_loader):
        # print(inputs['input_ids'].isnan().sum())
        inputs = {k: v.to(device) for k, v in inputs.items()}
        outputs = model(**inputs)
        outputs_masked = model_masked(**inputs)
    
        labels = inputs['labels'][..., 1:].contiguous()
        loss = outputs['loss'].view(labels.size())
        # print('loss',loss.isnan().sum())
        loss_masked = outputs_masked['loss'].view(labels.size())
        # print('loss mask', loss_masked.isnan().sum())
        loss_diff = loss_masked - loss

        labels = labels.contiguous()    #[:, 256:]
        loss_diff = loss_diff.contiguous()  #[:, 256:]
        # print(loss_diff.isnan().sum())
        puncs = (labels.unsqueeze(-1) == MASKED_VOCAB.unsqueeze(0).unsqueeze(0)).sum(dim=-1) == 1
        puncs_diff += loss_diff[puncs].sum()
        puncs_diff_abs += loss_diff[puncs].abs().sum()
        puncs_num += puncs.sum()

        after_puncs_diff += loss_diff[:, 1:].contiguous()[puncs[:, :-1]].sum()
        after_puncs_diff_abs += loss_diff[:, 1:].contiguous()[puncs[:, :-1]].abs().sum()
        after_puncs_num += puncs[:, :-1].sum()

        nonpuncs = (labels.unsqueeze(-1) == MASKED_VOCAB.unsqueeze(0).unsqueeze(0)).sum(dim=-1) == 0
        total_diff += loss_diff[nonpuncs].sum()
        total_diff_abs += loss_diff[nonpuncs].abs().sum()
        total_num += nonpuncs.sum()
        # total_diff += loss_diff.sum()
        # total_diff_abs += loss_diff.abs().sum()
        # total_num += loss_diff.numel()

print(puncs_diff/puncs_num)
print(puncs_diff_abs/puncs_num)
print()
print(after_puncs_diff/after_puncs_num)
print(after_puncs_diff_abs/after_puncs_num)
print()
print(total_diff/total_num)
print(total_diff_abs/total_num)

empty cache...
empty cache...


  1%|▏         | 14/1088 [00:08<08:16,  2.16it/s]

layer 5 tensor(7.2457, device='cuda:0') tensor(117.2583, device='cuda:0') tensor(0.4737, device='cuda:0')
. . . . . . (e) . . . . (e) . (s) (crimin) (ate) (▁profession) . . ." : (▁and) (:) (▁Iron) (▁Market) . ." (g) (Yes) r . . . ." (▁train) (Yes) (r) . (.) ." ence (▁distress) . (t) (it) (▁world) . ▁ ▁and . (▁Iron) (▁Market) (▁and) (.) (a) (s) (.)


  1%|▏         | 15/1088 [00:08<08:14,  2.17it/s]

layer 4 tensor(4.9912, device='cuda:0') tensor(71.3122, device='cuda:0') tensor(0.2917, device='cuda:0')
. (▁Marble) . . . . . . . . ; . (;) . . . (▁Grand) (▁Canyon) . . (▁Little) (;) . (.)


...

layer 1 tensor(1.1185, device='cuda:0') tensor(60.7151, device='cuda:0') tensor(0.3182, device='cuda:0')
. . . . . . ▁dear . . (▁dear) . (der) (▁Sir) (.) . . . . (▁bank) (row) . (.)


 99%|█████████▊| 1072/1088 [08:26<00:07,  2.15it/s]

layer 2 tensor(3.4573, device='cuda:0') tensor(70.2686, device='cuda:0') tensor(0.7111, device='cuda:0')
▁gall . (▁Cad) (e) (t) ▁Mail (▁marriage) (▁Made) ▁ (Ann) (e) (bau) (▁mai) (▁Ru) (e) (▁Barb) . (▁King) (▁Francis) (▁Maj) (▁Ne) (polita) (▁Mail) (▁Court) (▁Pie) . (▁) (▁) (▁gall) (▁mein) (▁Pas) (chal) (▁) . . . ▁La val (.) ▁La val (▁emb) (▁La) (val) (▁summon)


100%|██████████| 1088/1088 [08:33<00:00,  2.12it/s]

tensor(0.0626, device='cuda:0')
tensor(0.0728, device='cuda:0')

tensor(0.0028, device='cuda:0')
tensor(0.0631, device='cuda:0')

tensor(-0.0051, device='cuda:0')
tensor(0.0603, device='cuda:0')


## Loss Difference with respect to all cached tokens

In [4]:
# calculate frequency
token_freq = torch.zeros(model.config.vocab_size)
token_idx = torch.arange(model.config.vocab_size).long()
for i in torch.randperm(len(tokenized_dataset['train'])).tolist(): #[:1000]:
    for idx in tokenized_dataset['train'][i]['input_ids']:
        token_freq[idx] += 1

max_freq = token_freq.max()
token_freq[tokenizer.unk_token_id] = -1
freq1_mask = token_idx[token_freq >= (max_freq * 0.01)]
freq2_mask = token_idx[(token_freq < (max_freq * 0.01)) & (token_freq >= (max_freq * 0.001))]
freq3_mask = token_idx[(token_freq < (max_freq * 0.001)) & (token_freq >= 0)]
print(max_freq)

tensor(102551.)


In [28]:
with torch.no_grad():

    model.to(device)
    model.reset_cache()
    model.eval()

    model_masked.to(device)
    model_masked.reset_cache()
    model_masked.eval()

    freq1_diff = 0
    freq1_num = 0
    freq2_diff = 0
    freq2_num = 0
    freq3_diff = 0
    freq3_num = 0
    freq4_diff = 0
    freq4_num = 0

    total_diff = 0
    total_num = 0

    for inputs in tqdm(valid_loader):
        inputs = {k: v.to(device) for k, v in inputs.items()}
        outputs = model(**inputs)
        outputs_masked = model_masked(**inputs)
    
        labels = inputs['labels'][..., 1:].contiguous()
        loss = outputs['loss'].view(labels.size())
        loss_masked = outputs_masked['loss'].view(labels.size())
        loss_diff = loss_masked - loss

        mask1 = (labels.unsqueeze(-1) == freq1_mask.to(device).unsqueeze(0).unsqueeze(0)).sum(dim=-1) > 0
        freq1_diff += loss_diff[mask1].sum()
        freq1_num += mask1.sum()

        mask2 = (labels.unsqueeze(-1) == freq2_mask.to(device).unsqueeze(0).unsqueeze(0)).sum(dim=-1) > 0
        freq2_diff += loss_diff[mask2].sum()
        freq2_num += mask2.sum()

        mask3 = (labels.unsqueeze(-1) == freq3_mask.to(device).unsqueeze(0).unsqueeze(0)).sum(dim=-1) > 0
        freq3_diff += loss_diff[mask3].sum()
        freq3_num += mask3.sum()

        mask4 = labels == tokenizer.unk_token_id
        freq4_diff += loss_diff[mask4].sum()
        freq4_num += mask4.sum()

        total_diff += loss_diff.sum()
        total_num += loss_diff.numel()

print(freq1_diff/freq1_num)
print()
print(freq2_diff/freq2_num)
print()
print(freq3_diff/freq3_num)
print()
print(freq4_diff/freq4_num)
print()
print(total_diff/total_num)
print()


empty cache...
empty cache...


  0%|          | 3/1088 [00:01<09:32,  1.90it/s]

layer 3 tensor(10.2300, device='cuda:0') tensor(87.1300, device='cuda:0') tensor(0.5192, device='cuda:0')
(▁the) (▁the) (,) ▁ . ▁The . (▁) . (’) (s) . (▁The) (▁) ▁ (,) . ▁ . (▁and) (▁) . . . (▁) ▁ . (▁) . (▁) (▁) . (▁) (▁and) (▁) . (▁) (▁) (▁) (▁) . ▁ . (▁) . . (▁) . (▁) . (.) (▁)


  2%|▏         | 24/1088 [00:11<08:12,  2.16it/s]

layer 2 tensor(6.0407, device='cuda:0') tensor(83.0278, device='cuda:0') tensor(0.6800, device='cuda:0')
▁ ▁ . . (▁Rud) (▁) . (▁Cro) (p) (▁Grey) . (▁) ▁Jane (▁trou) (▁Le) (l) (k) ▁ (.") (▁) (bul) (.) (fern) (▁Jane) (master)


...

tensor([ 124., 1479., 1662.,  393.,  438.]) tensor([-16.3221, -11.7811,  -7.2401,  -2.6991,   1.8419,   6.3828])
layer: 0 mean log alpha: -5.291950702667236 std log alpha: 4.497146129608154 select ratio: 0.1123046875 sum ratio 0.0
▁sit s (,) * ▁the ▁soothe ▁for ▁to ▁say n (,) ▁* be fit s ▁Un to ▁ a ▁wo e ful ▁ w ight ▁ a ▁ d re ary ▁ fer e (,) * ▁* comp ani on ▁And ▁to ▁ a ▁sorry ▁tale ▁ a ▁sorry ▁cheer (.) * ▁* count en ance ▁For ▁I (,) ▁that ▁God (▁of) ▁Love ' s ▁servant s ▁serve (,) ▁Nor ▁dar e ▁to ▁love ▁for ▁mine ▁unlike lines s (,) * ▁ <unk> 3 > ▁* un suit able ness ▁Pra y e ▁for ▁speed (,) * ▁although ▁I ▁should


 92%|█████████▏| 997/1088 [07:53<00:42,  2.14it/s]

layer 1 tensor(1.3609, device='cuda:0') tensor(53.6366, device='cuda:0') tensor(0.3548, device='cuda:0')
▁ . . ▁and ▁and ▁and ▁ ▁ (l) ▁ ▁ (▁) ▁ ▁ ▁ (▁) . . (LIN) (.) ▁ ▁ . ▁ (▁) (▁) ▁and (.) (▁and) (▁and) (.)


100%|██████████| 1088/1088 [08:36<00:00,  2.10it/s]

tensor(0.0059, device='cuda:0')

tensor(0.0136, device='cuda:0')

tensor(0.0351, device='cuda:0')

tensor(0.0153, device='cuda:0')

tensor(0.0127, device='cuda:0')

